# Flask + ngrok demo

Run the cells top-to-bottom to:
1) start a local Flask app
2) expose it to the internet via ngrok

If ngrok asks for an authtoken, set `NGROK_AUTHTOKEN` in your environment (or paste it into the cell).

In [1]:
from kaggle_secrets import UserSecretsClient
import os


user_secrets = UserSecretsClient()
#secret_value_0 = user_secrets.get_secret("db_string")
os.environ["HF_TOKEN"] = hf_token = user_secrets.get_secret("HF_TOKEN")
#secret_value_2 = user_secrets.get_secret("mega_email")
#secret_value_3 = user_secrets.get_secret("mega_pass")
#secret_value_4 = user_secrets.get_secret("ngrok")
ngrok_2 = user_secrets.get_secret("ngrok2")



os.environ["NGROK_AUTHTOKEN"] = user_secrets.get_secret("ngrok2")

In [2]:
isCondaSet = False
isDA3Set = False

In [3]:
# (Optional) Install deps into the active Jupyter kernel environment
import sys

!{sys.executable} -m pip install -q flask pyngrok gdown mega.py imageio-ffmpeg opencv-python-headless

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
plotly 5.24.1 requires tenacity>=6.2.0, but you have tenacity 5.1.5 which is incompatible.
google-genai 1.55.0 requires tenacity<9.2.0,>=8.2.3, but you have tenacity 5.1.5 which is incompatible.
google-adk 1.21.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 5.1.5 which is incompatible.
langchain-core 1.2.7 requires tenacity!=8.4.0,<10.0.0,>=8.1.0, but you have tenacity 5.1.5 which is incompatible.


## Flask App Code

In [4]:
from flask import Flask, jsonify, render_template_string, request, send_from_directory, url_for
from threading import Thread, Lock, Event
from werkzeug.serving import make_server
from werkzeug.utils import secure_filename
from pathlib import Path
from collections import deque
from datetime import datetime
from queue import Queue, Empty, Full
import os
import shutil
import subprocess
import ipaddress
import socket
import time
import itertools
import urllib.parse
import urllib.request
import uuid
import cv2


TWELVE_HOURS_SEC = 12 * 60 * 60


def _get_process_start_time_epoch():
    # Best-effort kernel start timestamp (seconds since Unix epoch)
    try:
        import psutil  # type: ignore

        return float(psutil.Process(os.getpid()).create_time())
    except Exception:
        pass

    if os.name == "nt":
        try:
            import ctypes
            import ctypes.wintypes as wintypes

            class FILETIME(ctypes.Structure):
                _fields_ = [
                    ("dwLowDateTime", wintypes.DWORD),
                    ("dwHighDateTime", wintypes.DWORD),
                ]

            creation_time = FILETIME()
            exit_time = FILETIME()
            kernel_time = FILETIME()
            user_time = FILETIME()

            kernel32 = ctypes.WinDLL("kernel32", use_last_error=True)
            kernel32.GetCurrentProcess.restype = wintypes.HANDLE
            kernel32.GetProcessTimes.argtypes = [
                wintypes.HANDLE,
                ctypes.POINTER(FILETIME),
                ctypes.POINTER(FILETIME),
                ctypes.POINTER(FILETIME),
                ctypes.POINTER(FILETIME),
            ]
            kernel32.GetProcessTimes.restype = wintypes.BOOL

            handle = kernel32.GetCurrentProcess()
            ok = kernel32.GetProcessTimes(
                handle,
                ctypes.byref(creation_time),
                ctypes.byref(exit_time),
                ctypes.byref(kernel_time),
                ctypes.byref(user_time),
            )
            if not ok:
                raise ctypes.WinError(ctypes.get_last_error())

            ft = (creation_time.dwHighDateTime << 32) + creation_time.dwLowDateTime
            return (ft / 10_000_000) - 11644473600
        except Exception:
            pass

    return None


def _format_hhmm(seconds: float) -> str:
    seconds = max(0, int(seconds))
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    return f"{hours:02d}:{minutes:02d}"


# --- Web terminal (call web_print(...) from other cells) ---
if "WEB_PRINT_QUEUE" not in globals():
    WEB_PRINT_QUEUE = Queue(maxsize=5000)
if "WEB_TERMINAL_BUFFER" not in globals():
    WEB_TERMINAL_BUFFER = deque(maxlen=2000)
if "WEB_TERMINAL_LOCK" not in globals():
    WEB_TERMINAL_LOCK = Lock()


def web_print(msg):
    ts = datetime.now().strftime("%H:%M:%S")
    text = "" if msg is None else str(msg)
    lines = text.splitlines() or [""]
    for line in lines:
        entry = f"[{ts}] {line}"
        try:
            WEB_PRINT_QUEUE.put_nowait(entry)
        except Full:
            try:
                WEB_PRINT_QUEUE.get_nowait()
            except Empty:
                pass
            try:
                WEB_PRINT_QUEUE.put_nowait(entry)
            except Exception:
                pass


def _drain_web_print_queue(max_items: int = 500) -> int:
    drained = 0
    while drained < max_items:
        try:
            entry = WEB_PRINT_QUEUE.get_nowait()
        except Empty:
            break
        with WEB_TERMINAL_LOCK:
            WEB_TERMINAL_BUFFER.append(entry)
        drained += 1
    return drained


# --- Uploaded video queue (FIFO) ---
VIDEO_EXTENSIONS = {
    ".mp4",
    ".mov",
    ".mkv",
    ".avi",
    ".webm",
    ".flv",
    ".wmv",
    ".m4v",
    ".3gp",
    ".mpeg",
    ".mpg",
    ".dav",
}

if "VIDEO_UPLOAD_QUEUE" not in globals():
    VIDEO_UPLOAD_QUEUE = deque(maxlen=200)
if "VIDEO_QUEUE_LOCK" not in globals():
    VIDEO_QUEUE_LOCK = Lock()

# --- fps adjusted queue (prepared videos) ---
if "DA3_QUEUE" not in globals():
    DA3_QUEUE = deque(maxlen=200)
if "DA3_QUEUE_LOCK" not in globals():
    DA3_QUEUE_LOCK = Lock()
if "DA3_JOB_QUEUE" not in globals():
    DA3_JOB_QUEUE = Queue(maxsize=500)
if "DA3_SEEN" not in globals():
    DA3_SEEN = set()
if "DA3_SEEN_LOCK" not in globals():
    DA3_SEEN_LOCK = Lock()

# --- Frame queue (bounded frames on disk) ---
if "FRAME_QUEUE" not in globals():
    FRAME_QUEUE = deque(maxlen=100)
if "FRAME_QUEUE_LOCK" not in globals():
    FRAME_QUEUE_LOCK = Lock()
if "FRAME_QUEUE_FILL_LOCK" not in globals():
    FRAME_QUEUE_FILL_LOCK = Lock()
if "FRAME_VIDEO_STATE" not in globals():
    FRAME_VIDEO_STATE = {}
if "FRAME_VIDEO_STATE_LOCK" not in globals():
    FRAME_VIDEO_STATE_LOCK = Lock()
if "FRAME_VIDEO_ORDER" not in globals():
    FRAME_VIDEO_ORDER = deque()
if "FRAME_QUEUE_WAKE_EVENT" not in globals():
    FRAME_QUEUE_WAKE_EVENT = Event()
if "FRAME_QUEUE_STOP_EVENT" not in globals():
    FRAME_QUEUE_STOP_EVENT = Event()
if "FRAME_QUEUE_WORKER" not in globals():
    FRAME_QUEUE_WORKER = None

# --- DA3 process worker (multi-device) ---
if "DA3_PROCESS_WORKERS" not in globals():
    DA3_PROCESS_WORKERS = []
if "DA3_PROCESS_WORKERS_LOCK" not in globals():
    DA3_PROCESS_WORKERS_LOCK = Lock()
if "DA3_PROCESS_STOP_EVENT" not in globals():
    DA3_PROCESS_STOP_EVENT = Event()
if "DA3_PROCESS_START_EVENT" not in globals():
    DA3_PROCESS_START_EVENT = Event()
if "DA3_PROCESS_WORKER_MANAGER" not in globals():
    DA3_PROCESS_WORKER_MANAGER = None
if "DA3_BACKEND_WORKERS" not in globals():
    DA3_BACKEND_WORKERS = []
if "DA3_BACKEND_WORKERS_LOCK" not in globals():
    DA3_BACKEND_WORKERS_LOCK = Lock()
# --- MEGA upload queue ---
if "MEGA_UPLOAD_QUEUE" not in globals():
    MEGA_UPLOAD_QUEUE = Queue(maxsize=500)
if "MEGA_UPLOAD_QUEUE_LOCK" not in globals():
    MEGA_UPLOAD_QUEUE_LOCK = Lock()
if "MEGA_UPLOAD_WORKER" not in globals():
    MEGA_UPLOAD_WORKER = None
if "MEGA_UPLOAD_STOP_EVENT" not in globals():
    MEGA_UPLOAD_STOP_EVENT = Event()
if "MEGA_CLIENT" not in globals():
    MEGA_CLIENT = None


def _is_video_path(path: Path) -> bool:
    # Prefer extension check; fall back to lightweight header sniffing.
    if path.suffix.lower() in VIDEO_EXTENSIONS:
        return True

    try:
        with open(path, "rb") as f:
            head = f.read(64)
    except Exception:
        return False

    # MP4/MOV/3GP family: ....ftyp....
    if len(head) >= 8 and head[4:8] == b"ftyp":
        return True
    # Matroska/WebM
    if head.startswith(bytes.fromhex("1a45dfa3")):
        return True
    # AVI
    if len(head) >= 12 and head.startswith(b"RIFF") and head[8:12] in {b"AVI ", b"AVIX"}:
        return True
    # FLV
    if head.startswith(b"FLV"):
        return True
    # Ogg container (may include video)
    if head.startswith(b"OggS"):
        return True
    # ASF (WMV)
    if head.startswith(bytes.fromhex("3026b2758e66cf11a6d900aa0062ce6c")):
        return True
    # MPEG program/elementary stream (very rough)
    if head[:4] in {bytes.fromhex("000001ba"), bytes.fromhex("000001b3")}:
        return True

    return False


def _ffmpeg_exe() -> str:
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    try:
        import imageio_ffmpeg  # type: ignore

        exe = imageio_ffmpeg.get_ffmpeg_exe()
        if exe:
            return exe
    except Exception:
        pass
    raise RuntimeError("ffmpeg not found. Install ffmpeg and ensure it's on PATH.")


def convert_dav_to_mp4(dav_path: Path) -> Path:
    if dav_path.suffix.lower() != ".dav":
        return dav_path

    out_path = dav_path.with_suffix(".mp4")
    if out_path.exists():
        out_path = dav_path.with_name(f"{dav_path.stem}_{uuid.uuid4().hex[:8]}.mp4")

    ffmpeg = _ffmpeg_exe()
    attempts = [
        ("video+audio", ["-map", "0:v:0", "-map", "0:a?", "-c", "copy"]),
        ("video-only", ["-map", "0:v:0", "-c", "copy"]),
    ]

    last_err = ""
    for label, extra in attempts:
        out_path.unlink(missing_ok=True)
        cmd = [
            ffmpeg,
            "-y",
            "-hide_banner",
            "-loglevel",
            "error",
            "-fflags",
            "+genpts",
            "-i",
            str(dav_path),
            *extra,
            "-movflags",
            "+faststart",
            str(out_path),
        ]
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode == 0 and out_path.exists() and out_path.stat().st_size > 0:
            web_print(f"DAV->MP4 conversion ok ({label}): {out_path.name}")
            return out_path

        last_err = (proc.stderr or proc.stdout or "").strip()
        if len(last_err) > 3000:
            last_err = last_err[-3000:]

    out_path.unlink(missing_ok=True)
    raise RuntimeError(f"ffmpeg DAV->MP4 failed. {last_err}")

    return out_path


def convert_video_to_target_fps(src_path: Path, target_fps: int) -> Path:
    fps = int(target_fps)
    if fps < 1 or fps > 240:
        raise ValueError('Target FPS must be between 1 and 240.')

    base = secure_filename(src_path.stem) or 'video'
    base = base[:60]
    out_name = f'fpsadj_{uuid.uuid4().hex[:12]}_{base}_fps{fps}.mp4'
    out_path = UPLOAD_DIR / out_name

    ffmpeg = _ffmpeg_exe()
    attempts = [
        ('copy-audio', ['-c:a', 'copy']),
        ('aac-audio', ['-c:a', 'aac', '-b:a', '192k']),
    ]

    last_err = ''
    for label, audio_args in attempts:
        out_path.unlink(missing_ok=True)
        cmd = [
            ffmpeg,
            '-y',
            '-hide_banner',
            '-loglevel',
            'error',
            '-fflags',
            '+genpts',
            '-i',
            str(src_path),
            '-map',
            '0:v:0',
            '-map',
            '0:a?',
            '-vf',
            f'fps={fps}',
            '-c:v',
            'libx264',
            '-preset',
            'veryfast',
            '-crf',
            '0',
            *audio_args,
            '-movflags',
            '+faststart',
            str(out_path),
        ]
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode == 0 and out_path.exists() and out_path.stat().st_size > 0:
            web_print(f'FPS adjust ok ({label}): {out_path.name}')
            return out_path

        last_err = (proc.stderr or proc.stdout or '').strip()
        if len(last_err) > 3000:
            last_err = last_err[-3000:]

    out_path.unlink(missing_ok=True)
    raise RuntimeError(f'ffmpeg fps conversion failed. {last_err}')


def enqueue_da3_job(video_path: Path, source: str = '') -> None:
    fps = get_target_fps()
    key = (str(video_path), int(fps))
    with DA3_SEEN_LOCK:
        if key in DA3_SEEN:
            return
        DA3_SEEN.add(key)

    job = {
        'input_path': str(video_path),
        'input_stored_name': video_path.name,
        'target_fps': int(fps),
        'ts': float(time.time()),
        'source': str(source or ''),
    }
    try:
        DA3_JOB_QUEUE.put_nowait(job)
    except Full:
        with DA3_SEEN_LOCK:
            DA3_SEEN.discard(key)
        web_print(f'FPS adjust job queue full; dropped: {video_path.name}')
        return

    web_print(f'FPS adjust job queued: {video_path.name} -> {fps}fps')


def _da3_worker_loop():
    web_print('FPS adjust worker started')
    while True:
        try:
            job = DA3_JOB_QUEUE.get(timeout=1)
        except Empty:
            continue
        if job is None:
            break

        src_path = Path(job.get('input_path', ''))
        fps = int(job.get('target_fps', 15))
        try:
            out_path = convert_video_to_target_fps(src_path, fps)
            try:
                size_bytes = out_path.stat().st_size
            except Exception:
                size_bytes = 0

            item = {
                'stored_name': out_path.name,
                'path': str(out_path),
                'size_bytes': int(size_bytes),
                'size_mb': round(size_bytes / (1024 * 1024), 2),
                'ts': float(time.time()),
                'source': job.get('source', ''),
                'target_fps': fps,
                'input_stored_name': job.get('input_stored_name', src_path.name),
                'input_path': job.get('input_path', str(src_path)),
            }

            dropped = None
            with DA3_QUEUE_LOCK:
                if DA3_QUEUE.maxlen and len(DA3_QUEUE) >= DA3_QUEUE.maxlen:
                    dropped = DA3_QUEUE[0]
                DA3_QUEUE.append(item)

            if dropped:
                web_print(f"FPS adjusted queue full; dropped oldest: {dropped.get('stored_name')}")
            web_print(f"FPS adjusted queued: {out_path.name}")

            try:
                register_video_for_frame_queue(out_path, source=job.get('source', ''))
            except Exception as exc:
                web_print(f"Frame queue register failed for {out_path.name}: {exc}")
        except Exception as exc:
            web_print(f"FPS adjust failed for {src_path.name}: {exc}")


def start_da3_worker():
    global DA3_WORKER
    try:
        if DA3_WORKER is not None and DA3_WORKER.is_alive():
            return
    except Exception:
        pass
    DA3_WORKER = Thread(target=_da3_worker_loop, daemon=True, name='FPS-Adjust-Worker')
    DA3_WORKER.start()


def da3_queue_list():
    with DA3_QUEUE_LOCK:
        return list(DA3_QUEUE)


def da3_queue_pop():
    with DA3_QUEUE_LOCK:
        if not DA3_QUEUE:
            return None
        return DA3_QUEUE.popleft()


def _is_da3_set() -> bool:
    try:
        return bool(globals().get("isDA3Set", False))
    except Exception:
        return False


def register_video_for_frame_queue(video_path: Path, video_stored_name: str = "", source: str = "") -> None:
    if not video_stored_name:
        video_stored_name = video_path.name
    key = str(video_path.resolve())

    with FRAME_VIDEO_STATE_LOCK:
        if key in FRAME_VIDEO_STATE:
            return

        base = secure_filename(video_path.stem) or 'video'
        base = base[:60]
        digest = uuid.uuid5(uuid.NAMESPACE_URL, key).hex[:10]
        frames_dir = FRAMES_DIR / f'{base}_{digest}'

        FRAME_VIDEO_STATE[key] = {
            'video_path': key,
            'video_stored_name': str(video_stored_name),
            'video_file_url': f'/uploads/{video_stored_name}',
            'frames_dir': str(frames_dir),
            'next_frame_idx': 0,
            'frame_count': None,
            'report_every': 100,
            'next_report': 100,
            'ts': float(time.time()),
            'source': str(source or ''),
        }
        FRAME_VIDEO_ORDER.append(key)

    web_print(f'Frame queue: registered video: {video_path.name}')
    start_frame_queue_worker()
    FRAME_QUEUE_WAKE_EVENT.set()


def _opencv_extract_frame(video_path: Path, frame_idx: int, out_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        raise RuntimeError('OpenCV failed to open video')

    try:
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        if frame_count <= 0:
            frame_count = None

        cap.set(cv2.CAP_PROP_POS_FRAMES, float(frame_idx))
        ok, frame = cap.read()
    finally:
        cap.release()

    if not ok or frame is None:
        return False, frame_count

    out_path.parent.mkdir(parents=True, exist_ok=True)
    ok2 = cv2.imwrite(str(out_path), frame, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    if not ok2:
        raise RuntimeError('Failed to write frame image')
    return True, frame_count


def _next_frame_task_for_video(video_key: str):
    with FRAME_VIDEO_STATE_LOCK:
        state = FRAME_VIDEO_STATE.get(video_key)
        if not state:
            return None
        video_path = Path(state.get('video_path', ''))
        video_stored_name = state.get('video_stored_name', '') or video_path.name
        frames_dir = Path(state.get('frames_dir', ''))
        next_idx = int(state.get('next_frame_idx', 0))
        frame_count = state.get('frame_count', None)

    if not video_path.exists():
        raise FileNotFoundError(f'Video not found: {video_path}')

    if frame_count is not None and next_idx >= int(frame_count):
        return None

    out_path = frames_dir / f'frame_{next_idx:06d}.png'
    if not out_path.exists():
        ok, fc = _opencv_extract_frame(video_path, next_idx, out_path)
        if fc is not None:
            frame_count = int(fc)
        if not ok:
            with FRAME_VIDEO_STATE_LOCK:
                st = FRAME_VIDEO_STATE.get(video_key)
                if st is not None and frame_count is not None:
                    st['frame_count'] = int(frame_count)
                    st['done'] = True
            return None

    rel = out_path.relative_to(FRAMES_DIR).as_posix()
    task = {
        'video_file_url': f'/uploads/{video_stored_name}',
        'frame_no': int(next_idx),
        'frame_download_url': f'/frames/{rel}',
        'frame_local_path': str(out_path.resolve()),
    }

    msg = None
    with FRAME_VIDEO_STATE_LOCK:
        st = FRAME_VIDEO_STATE.get(video_key)
        if st is not None:
            if frame_count is not None:
                st['frame_count'] = int(frame_count)
                if st.get('report_every', 100) == 100:
                    rep = max(10, int(frame_count) // 20) if int(frame_count) else 100
                    st['report_every'] = int(rep)
                    st['next_report'] = int(rep)

            st['next_frame_idx'] = int(next_idx) + 1
            extracted = int(st['next_frame_idx'])
            fc2 = st.get('frame_count', None)
            rep = int(st.get('report_every', 100) or 100)
            nxt = int(st.get('next_report', rep) or rep)

            if extracted == 1:
                msg = f'Frame queue: started {video_path.name}'
            elif extracted >= nxt:
                if fc2:
                    pct = int((extracted / int(fc2)) * 100)
                    msg = f'Frame queue: {video_path.name} {extracted}/{int(fc2)} ({pct}%)'
                else:
                    msg = f'Frame queue: {video_path.name} extracted {extracted}'
                st['next_report'] = int(nxt) + int(rep)

            if fc2 and extracted >= int(fc2):
                st['done'] = True
                msg = f'Frame queue: done {video_path.name} ({extracted}/{int(fc2)})'

    if msg:
        web_print(msg)

    return task


def _fill_frame_queue(max_new: int = 200) -> int:
    try:
        max_new = int(max_new)
    except Exception:
        max_new = 200
    max_new = max(1, min(max_new, 2000))

    filled = 0
    with FRAME_QUEUE_FILL_LOCK:
        while filled < max_new:
            with FRAME_QUEUE_LOCK:
                max_len = FRAME_QUEUE.maxlen or 0
                if max_len and len(FRAME_QUEUE) >= max_len:
                    break

            with FRAME_VIDEO_STATE_LOCK:
                if not FRAME_VIDEO_ORDER:
                    break
                key = FRAME_VIDEO_ORDER.popleft()

            try:
                task = _next_frame_task_for_video(key)
            except Exception as exc:
                with FRAME_VIDEO_STATE_LOCK:
                    FRAME_VIDEO_STATE.pop(key, None)
                web_print(f'Frame queue: dropped video due to error: {exc}')
                continue

            if not task:
                with FRAME_VIDEO_STATE_LOCK:
                    FRAME_VIDEO_STATE.pop(key, None)
                continue

            with FRAME_QUEUE_LOCK:
                max_len = FRAME_QUEUE.maxlen or 0
                if (not max_len) or len(FRAME_QUEUE) < max_len:
                    FRAME_QUEUE.append(task)
                    filled += 1

            with FRAME_VIDEO_STATE_LOCK:
                st = FRAME_VIDEO_STATE.get(key)
                if st and not st.get('done', False):
                    FRAME_VIDEO_ORDER.append(key)
                else:
                    FRAME_VIDEO_STATE.pop(key, None)

    return int(filled)


def _frame_queue_worker_loop():
    web_print('Frame queue worker started')
    while True:
        if FRAME_QUEUE_STOP_EVENT.is_set():
            break
        try:
            filled = _fill_frame_queue(max_new=200)
        except Exception as exc:
            filled = 0
            web_print(f'Frame queue worker error: {exc}')
        if filled <= 0:
            FRAME_QUEUE_WAKE_EVENT.wait(timeout=0.5)
            FRAME_QUEUE_WAKE_EVENT.clear()


def start_frame_queue_worker():
    global FRAME_QUEUE_WORKER
    try:
        if FRAME_QUEUE_WORKER is not None and FRAME_QUEUE_WORKER.is_alive():
            return
    except Exception:
        pass
    try:
        FRAME_QUEUE_STOP_EVENT.clear()
    except Exception:
        pass
    FRAME_QUEUE_WORKER = Thread(target=_frame_queue_worker_loop, daemon=True, name='Frame-Queue-Worker')
    FRAME_QUEUE_WORKER.start()


def da3_task_queue_peek(limit: int = 200):
    try:
        limit = int(limit)
    except Exception:
        limit = 200
    limit = max(1, min(limit, 2000))

    with FRAME_QUEUE_LOCK:
        items = list(itertools.islice(reversed(FRAME_QUEUE), limit))
    items.reverse()
    return items


def da3_task_queue_pop():
    with FRAME_QUEUE_LOCK:
        if not FRAME_QUEUE:
            return None
        item = FRAME_QUEUE.popleft()
    FRAME_QUEUE_WAKE_EVENT.set()
    return item


# --- DA3 Processing Worker (multi-device) ---

def _run_da3_subprocess(frame_task, device_no, backend_port_no):
    """Run DA3 subprocess for a single frame task"""
    frame_path = frame_task.get('frame_local_path', '')
    video_file_url = frame_task.get('video_file_url', '')
    frame_no = frame_task.get('frame_no', 0)
    
    # Extract video name and frame name from paths
    video_name = Path(video_file_url).stem if video_file_url else "unknown"
    file_name = f"frame_{frame_no:06d}"

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = f"{device_no}"
    env["DA3_BACKEND_PORT"] = str(backend_port_no)
    
    cmd = [
        "bash", "on_conda",
        "da3", "auto", frame_path,
        "--export-format", "glb-feat_vis",
        "--export-dir", f"output/{video_name}/{file_name}",
        "--auto-cleanup",
        '--backend-url',
        'http://localhost:{backend_port_no}'
    ]
    
    web_print(f"[Device {device_no}] Starting DA3: {file_name} (backend {backend_port_no})")
    
    try:
        process = subprocess.Popen(
            cmd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            stdin=subprocess.PIPE,
            universal_newlines=True,
            bufsize=1
        )
        
        if process.stdout:
            for line in process.stdout:
                line = line.strip()
                if line:
                    web_print(f"[Device {device_no}] {line}")
        
        process.wait()
        
        if process.returncode == 0:
            web_print(f"[Device {device_no}] Completed: {file_name}")
            if isMEGACredSet:
                output_dir = f"output/{video_name}/{file_name}"
                enqueue_mega_upload(video_name, file_name, output_dir, frame_path)
        else:

            web_print(f"[Device {device_no}] Failed (code {process.returncode}): {file_name}")
            
    except Exception as exc:
        web_print(f"[Device {device_no}] Error: {exc}")


def _da3_backend_worker(device_no, backend_port_no):
    """Backend worker that hosts the DA3 backend server for a device"""
    web_print(f"[Backend {device_no}] Starting backend on port {backend_port_no}")

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = f"{device_no}"

    cmd = [
        "bash", "on_conda",
        "da3", "backend",
        "--model-dir", "depth-anything/DA3NESTED-GIANT-LARGE",
        "--gallery-dir", "gallery",
        "--host", "0.0.0.0",
        "--port", str(backend_port_no),
    ]

    try:
        process = subprocess.Popen(
            cmd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            stdin=subprocess.PIPE,
            universal_newlines=True,
            bufsize=1
        )

        if process.stdout:
            for line in process.stdout:
                if DA3_PROCESS_STOP_EVENT.is_set():
                    break
                line = line.strip()
                if line:
                    web_print(f"[Backend {device_no}] {line}")

        if DA3_PROCESS_STOP_EVENT.is_set():
            try:
                process.terminate()
            except Exception:
                pass

        process.wait()

        if process.returncode == 0:
            web_print(f"[Backend {device_no}] Backend stopped")
        else:
            web_print(f"[Backend {device_no}] Backend exited with code {process.returncode}")

    except Exception as exc:
        web_print(f"[Backend {device_no}] Error: {exc}")

def _da3_sub_worker(device_no):
    """Sub-worker for a specific device that processes frame tasks"""
    web_print(f"[Device {device_no}] Sub-worker started, waiting for DA3...")
    
    # Wait until isDA3Set = True
    while not _is_da3_set():
        if DA3_PROCESS_STOP_EVENT.is_set():
            return
        DA3_PROCESS_START_EVENT.wait(timeout=1.0)
    
    web_print(f"[Device {device_no}] DA3 ready, starting processing")
    backend_port_no = 8008 + int(device_no)
    
    while True:
        if DA3_PROCESS_STOP_EVENT.is_set():
            web_print(f"[Device {device_no}] Sub-worker stopping")
            break
        
        # Get a task from the frame queue
        task = da3_task_queue_pop()
        
        if task is None:
            # No task available, wait a bit
            time.sleep(0.5)
            continue
        
        # Process the task
        _run_da3_subprocess(task, device_no, backend_port_no)


def _da3_process_worker_manager():
    """Main worker that manages sub-workers for each device"""
    web_print("DA3 process worker manager started")
    
    # Wait until isDA3Set = True
    while not _is_da3_set():
        if DA3_PROCESS_STOP_EVENT.is_set():
            return
        DA3_PROCESS_START_EVENT.wait(timeout=1.0)
    
    web_print("DA3 is set, spawning backend workers")

    device_count = get_device_count()
    with DA3_BACKEND_WORKERS_LOCK:
        while len(DA3_BACKEND_WORKERS) < device_count:
            device_no = len(DA3_BACKEND_WORKERS)
            backend_port_no = 8008 + device_no
            worker = Thread(
                target=_da3_backend_worker,
                args=(device_no, backend_port_no),
                daemon=True,
                name=f"backend_worker_{device_no}"
            )
            worker.start()
            DA3_BACKEND_WORKERS.append(worker)
            web_print(f"Started backend worker for device {device_no} on port {backend_port_no}")

    web_print("Waiting 15s for backend workers to warm up")
    if DA3_PROCESS_STOP_EVENT.wait(timeout=15.0):
        return

    web_print("DA3 is set, spawning sub-workers")
    
    while True:
        if DA3_PROCESS_STOP_EVENT.is_set():
            web_print("DA3 process worker manager stopping")
            break
        
        # Get current device count
        device_count = get_device_count()
        device_nos = list(range(device_count))
        
        with DA3_PROCESS_WORKERS_LOCK:
            # Stop workers that exceed current device count
            while len(DA3_PROCESS_WORKERS) > device_count:
                worker = DA3_PROCESS_WORKERS.pop()
                # Workers will stop via stop event
            
            # Start new workers if needed
            while len(DA3_PROCESS_WORKERS) < device_count:
                device_no = len(DA3_PROCESS_WORKERS)
                worker = Thread(
                    target=_da3_sub_worker,
                    args=(device_no,),
                    daemon=True,
                    name=f'DA3-Sub-Worker-{device_no}'
                )
                worker.start()
                DA3_PROCESS_WORKERS.append(worker)
                web_print(f"Started DA3 sub-worker for device {device_no}")
        
        # Sleep and check for device count changes
        time.sleep(2.0)


def start_da3_process_workers():
    """Start the DA3 process worker manager"""
    global DA3_PROCESS_WORKER_MANAGER
    
    try:
        if DA3_PROCESS_WORKER_MANAGER is not None and DA3_PROCESS_WORKER_MANAGER.is_alive():
            return
    except Exception:
        pass
    
    try:
        DA3_PROCESS_STOP_EVENT.clear()
    except Exception:
        pass
    
    DA3_PROCESS_WORKER_MANAGER = Thread(
        target=_da3_process_worker_manager,
        daemon=True,
        name='DA3-Process-Worker-Manager'
    )
    DA3_PROCESS_WORKER_MANAGER.start()
    web_print("DA3 process worker manager started")


def stop_da3_process_workers():
    """Stop all DA3 process workers"""
    web_print("Stopping DA3 process workers...")
    
    DA3_PROCESS_STOP_EVENT.set()
    DA3_PROCESS_START_EVENT.set()  # Wake up any waiting workers
    
    with DA3_PROCESS_WORKERS_LOCK:
        for worker in DA3_PROCESS_WORKERS:
            try:
                worker.join(timeout=2.0)
            except Exception:
                pass
        DA3_PROCESS_WORKERS.clear()
    with DA3_BACKEND_WORKERS_LOCK:
        for worker in DA3_BACKEND_WORKERS:
            try:
                worker.join(timeout=2.0)
            except Exception:
                pass
        DA3_BACKEND_WORKERS.clear()
    
    try:
        if DA3_PROCESS_WORKER_MANAGER is not None:
            DA3_PROCESS_WORKER_MANAGER.join(timeout=2.0)
    except Exception:
        pass
    
    web_print("DA3 process workers stopped")


def enqueue_video(path: Path, source: str = "") -> None:
    original_path = path
    if path.suffix.lower() == ".dav":
        web_print(f"Converting DAV to MP4: {path.name}")
        try:
            path = convert_dav_to_mp4(path)
        except Exception as exc:
            web_print(f"DAV->MP4 conversion failed: {exc}")
            return
        web_print(f"Converted DAV -> MP4: {path.name}")
        source = (source + " (dav->mp4)") if source else "dav->mp4"

    if not _is_video_path(path):
        web_print(f"[enqueue_video] : {path} is not a video path")
        return

    try:
        size_bytes = path.stat().st_size
    except Exception:
        size_bytes = 0

    item = {
        "stored_name": path.name,
        "path": str(path),
        "size_bytes": int(size_bytes),
        "size_mb": round(size_bytes / (1024 * 1024), 2),
        "ts": float(time.time()),
        "source": str(source or ""),
        "original_stored_name": original_path.name if original_path != path else "",
        "original_path": str(original_path) if original_path != path else "",
    }

    dropped = None
    with VIDEO_QUEUE_LOCK:
        if VIDEO_UPLOAD_QUEUE.maxlen and len(VIDEO_UPLOAD_QUEUE) >= VIDEO_UPLOAD_QUEUE.maxlen:
            dropped = VIDEO_UPLOAD_QUEUE[0]
        VIDEO_UPLOAD_QUEUE.append(item)

    if dropped:
        web_print(f"Video queue full; dropped oldest: {dropped.get('stored_name')}")

    src = f" ({source})" if source else ""
    web_print(f"Enqueued video{src}: {item['stored_name']} ({item['size_mb']} MB)")

    try:
        enqueue_da3_job(path, source=source)
    except Exception as exc:
        web_print(f"FPS adjust enqueue failed for {path.name}: {exc}")


def video_queue_list():
    with VIDEO_QUEUE_LOCK:
        return list(VIDEO_UPLOAD_QUEUE)


def video_queue_pop():
    with VIDEO_QUEUE_LOCK:
        if not VIDEO_UPLOAD_QUEUE:
            return None
        return VIDEO_UPLOAD_QUEUE.popleft()


def _is_port_free(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.2)
        return sock.connect_ex((host, port)) != 0


def _pick_port(host: str, preferred: int = 5000) -> int:
    if _is_port_free(host, preferred):
        return preferred
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind((host, 0))
        return int(sock.getsockname()[1])


class _ServerThread(Thread):
    def __init__(self, app: Flask, host: str, port: int):
        super().__init__(daemon=True)
        self._server = make_server(host, port, app)
        self._ctx = app.app_context()
        self._ctx.push()

    def run(self):
        self._server.serve_forever()

    def shutdown(self):
        self._server.shutdown()


# If you re-run this cell, stop the previous server first.
if "server" in globals():
    try:
        server.shutdown()
    except Exception:
        pass

# Persist kernel start time across cell re-runs.
if "_KERNEL_START_TS" not in globals():
    _KERNEL_START_TS = _get_process_start_time_epoch() or time.time()
KERNEL_START_TS = float(_KERNEL_START_TS)

# Persist settings across cell re-runs.
if "SETTINGS_LOCK" not in globals():
    SETTINGS_LOCK = Lock()
if "_TARGET_FPS" not in globals():
    _TARGET_FPS = 15
if "_DEVICE_COUNT" not in globals():
    _DEVICE_COUNT = 2
if "_MEGA_EMAIL" not in globals():
    _MEGA_EMAIL = ""
if "_MEGA_PASSWORD" not in globals():
    _MEGA_PASSWORD = ""
if "isMEGACredSet" not in globals():
    isMEGACredSet = False


def get_target_fps() -> int:
    with SETTINGS_LOCK:
        try:
            return int(_TARGET_FPS)
        except Exception:
            return 15


def set_target_fps(value) -> int:
    global _TARGET_FPS
    fps = int(value)
    if fps < 1 or fps > 240:
        raise ValueError("Target FPS must be between 1 and 240.")
    with SETTINGS_LOCK:
        _TARGET_FPS = fps
    return fps


def get_device_count() -> int:
    with SETTINGS_LOCK:
        try:
            return int(_DEVICE_COUNT)
        except Exception:
            return 2


def set_device_count(value) -> int:
    global _DEVICE_COUNT
    count = int(value)
    if count < 1 or count > 8:
        raise ValueError("Device count must be between 1 and 8.")
    with SETTINGS_LOCK:
        _DEVICE_COUNT = count
    return count


def _patch_asyncio_coroutine():
    try:
        import asyncio
        import types
    except Exception:
        return
    if not hasattr(asyncio, "coroutine"):
        asyncio.coroutine = types.coroutine

def set_mega_credentials(email: str, password: str) -> bool:
    global _MEGA_EMAIL, _MEGA_PASSWORD, isMEGACredSet, MEGA_CLIENT
    if not email or not password:
        return False
    with SETTINGS_LOCK: 
        _MEGA_EMAIL = email
        _MEGA_PASSWORD = password
        isMEGACredSet = True
    try:
        _patch_asyncio_coroutine()
        from mega import Mega
        MEGA_CLIENT = Mega().login(email, password)
        web_print("MEGA.nz login successful")
        return True
    except Exception as exc:
        web_print(f"MEGA.nz login failed: {exc}")
        isMEGACredSet = False
        return False


def get_mega_client():
    global MEGA_CLIENT, isMEGACredSet
    if not isMEGACredSet:
        return None
    if MEGA_CLIENT is not None:
        return MEGA_CLIENT
    try:
        _patch_asyncio_coroutine()
        from mega import Mega
        with SETTINGS_LOCK:
            MEGA_CLIENT = Mega().login(_MEGA_EMAIL, _MEGA_PASSWORD)
        return MEGA_CLIENT
    except Exception as exc:
        web_print(f"MEGA client creation failed: {exc}")
        return False





def enqueue_mega_upload(video_name, file_name, output_dir, frame_path):
    item = {
        "video_name": video_name,
        "file_name": file_name,
        "output_dir": output_dir,
        "frame_path": frame_path,
        "ts": time.time()
    }
    try:
        MEGA_UPLOAD_QUEUE.put_nowait(item)
    except Full:
        web_print(f"MEGA upload queue full; dropped: {video_name}/{file_name}")
        return
    web_print(f"Enqueued for MEGA upload: {video_name}/{file_name}")


def _zip_directory(src_dir, zip_path):
    import zipfile
    src_path = Path(src_dir)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file in src_path.rglob('*'):
            if file.is_file():
                arcname = file.relative_to(src_path)
                zipf.write(file, arcname)
    return zip_path


def _get_or_create_mega_folder(client, folder_path):
    folders = client.get_files()
    parts = folder_path.strip('/').split('/')
    current_parent = None
    for part in parts:
        found = None
        for f in folders:
            if f.get('type') == 'folder' and f.get('name') == part and f.get('parent_id') == current_parent:
                found = f
                break
        if not found:
            try:
                found = client.create_folder(part, parent=current_parent)
                folders = client.get_files()
            except Exception as exc:
                web_print(f"Failed to create MEGA folder {part}: {exc}")
                return None
        current_parent = found['id']
    return current_parent


def _upload_to_mega(file_path, mega_path):
    client = get_mega_client()
    if not client:
        raise RuntimeError("MEGA client not available")
    folder_id = _get_or_create_mega_folder(client, mega_path)
    if not folder_id:
        raise RuntimeError(f"Could not get/create MEGA folder: {mega_path}")
    uploaded = client.upload(str(file_path), dest=folder_id)
    return uploaded


def _mega_upload_worker():
    web_print("MEGA upload worker started")
    while not isMEGACredSet:
        if MEGA_UPLOAD_STOP_EVENT.is_set():
            return
        time.sleep(1.0)
    web_print("MEGA credentials set, starting uploads")
    while True:
        if MEGA_UPLOAD_STOP_EVENT.is_set():
            web_print("MEGA upload worker stopping")
            break
        try:
            item = MEGA_UPLOAD_QUEUE.get(timeout=1.0)
        except:
            continue
        video_name = item.get("video_name", "unknown")
        file_name = item.get("file_name", "unknown")
        output_dir = item.get("output_dir", "")
        frame_path = item.get("frame_path", "")
        try:
            output_path = Path(output_dir)
            zip_path = output_path.with_suffix('.zip')
            if output_path.exists():
                web_print(f"Zipping: {output_dir}")
                _zip_directory(output_path, zip_path)
                web_print(f"Created zip: {zip_path.name}")
            else:
                web_print(f"Output directory not found: {output_dir}")
                continue
            mega_folder_path = f"da3/{video_name}"
            web_print(f"Uploading to MEGA: {mega_folder_path}/{zip_path.name}")
            _upload_to_mega(zip_path, mega_folder_path)
            web_print(f"Uploaded to MEGA: {video_name}/{file_name}")
            import shutil
            shutil.rmtree(output_path, ignore_errors=True)
            web_print(f"Deleted output dir: {output_dir}")
            frame_file = Path(frame_path)
            if frame_file.exists():
                frame_file.unlink()
                web_print(f"Deleted frame: {frame_path}")
        except Exception as exc:
            web_print(f"MEGA upload failed for {video_name}/{file_name}: {exc}")


def start_mega_upload_worker():
    global MEGA_UPLOAD_WORKER
    try:
        if MEGA_UPLOAD_WORKER is not None and MEGA_UPLOAD_WORKER.is_alive():
            return
    except Exception:
        pass
    try:
        MEGA_UPLOAD_STOP_EVENT.clear()
    except Exception:
        pass
    MEGA_UPLOAD_WORKER = Thread(target=_mega_upload_worker, daemon=True, name='MEGA-Upload-Worker')
    MEGA_UPLOAD_WORKER.start()
    web_print("MEGA upload worker started")


def stop_mega_upload_worker():
    web_print("Stopping MEGA upload worker...")
    MEGA_UPLOAD_STOP_EVENT.set()
    try:
        if MEGA_UPLOAD_WORKER is not None:
            MEGA_UPLOAD_WORKER.join(timeout=2.0)
    except Exception:
        pass
    web_print("MEGA upload worker stopped")


app = Flask(__name__)
UPLOAD_DIR = Path("outputs/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_DIR = UPLOAD_DIR / "frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("/kaggle/working/output")
start_da3_worker()
start_frame_queue_worker()
start_da3_process_workers()
start_mega_upload_worker()

# On cell re-runs, also prepare already-queued videos.
try:
    with VIDEO_QUEUE_LOCK:
        _existing = list(VIDEO_UPLOAD_QUEUE)
    for _it in _existing:
        try:
            _p = Path(_it.get('path', ''))
            if _p and _p.exists():
                enqueue_da3_job(_p, source=_it.get('source', ''))
        except Exception:
            pass
except Exception:
    pass

# On cell re-runs, also build tasks for already FPS-adjusted videos.
try:
    with DA3_QUEUE_LOCK:
        _fps_existing = list(DA3_QUEUE)
    for _it in _fps_existing:
        try:
            _p = Path(_it.get('path', ''))
            if _p and _p.exists():
                register_video_for_frame_queue(_p, source=_it.get('source', ''))
        except Exception:
            pass
except Exception:
    pass
app.config["MAX_CONTENT_LENGTH"] = 25 * 1024 * 1024 * 1024  # 25GB


_HOME_HTML = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>Flask + ngrok demo</title>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bulma@1.0.2/css/bulma.min.css">
    <style>
        html, body {
            height: 100%;
            margin: 0;
            display: flex;
            flex-direction: column;
        }

        /* Navbar fixed height 10% */
        .navbar {
            height: 10vh;
            min-height: 3rem;
            display: flex;
            align-items: center;
        }

        /* Main section fills remaining height, uses flex column */
        .main-section {
            flex: 1;
            display: flex;
            flex-direction: column;
            padding: 0.75rem;
            gap: 0.75rem;
            min-height: 0;
        }

        /* Top row: forms (height 40% of remaining space) */
        .top-row {
            height: 40%;
            min-height: 0;
            display: flex;
            gap: 0.75rem;
        }
        .top-row .column {
            height: 100%;
            overflow-y: auto;
        }

        /* Bottom row: queue + terminal (takes remaining height) */
        .bottom-row {
            flex: 1;
            min-height: 0;
            display: flex;
            gap: 0.75rem;
        }
        .bottom-row .column {
            height: 100%;
            display: flex;
            flex-direction: column;
        }
        .bottom-row .box {
            flex: 1;
            min-height: 0;
            display: flex;
            flex-direction: column;
        }

        .scrollable-content {
            overflow-y: auto;
            flex: 1;
            min-height: 0;
        }

        .terminal-pre {
            background: #0b0f0b;
            color: #d1f7c4;
            padding: 12px;
            border-radius: 6px;
            font-family: monospace;
            font-size: 13px;
            white-space: pre-wrap;
            height: 100%;
            margin: 0;
        }

        .small { font-size: 0.85rem; color: #666; }

        /* Column widths */
        .col-30 { width: 30%; flex-shrink: 0; }
        .col-40 { width: 40%; flex-shrink: 0; }
        .col-70 { width: 70%; flex-shrink: 0; }

        /* Two equal sub-columns inside the 60% area */
        .top-row-left {
            display: flex;
            gap: 0.75rem;
            flex: 1;       /* takes the space not used by col-40 */
            min-width: 0;
            height: 100%;
        }
        .top-row-left .sub-column {
            flex: 1;
            min-width: 0;
            height: 100%;
            overflow-y: auto;
        }

        /* ── Toast notifications ─────────────────────────── */
        #toast-container {
            position: fixed;
            top: 1rem;
            right: 1rem;
            z-index: 9999;
            display: flex;
            flex-direction: column;
            gap: 0.5rem;
            pointer-events: none;
        }
        .toast-item {
            pointer-events: auto;
            min-width: 260px;
            max-width: 420px;
            opacity: 0;
            transform: translateX(20px);
            transition: opacity 0.25s ease, transform 0.25s ease;
            box-shadow: 0 4px 16px rgba(0,0,0,0.15);
        }
        .toast-item.toast-visible {
            opacity: 1;
            transform: translateX(0);
        }
        .toast-item.toast-hiding {
            opacity: 0;
            transform: translateX(20px);
        }
        .toast-item .delete {
            pointer-events: auto;
        }
    </style>
</head>
<body>

    <!-- Toast container (rendered outside normal flow) -->
    <div id="toast-container"></div>

    <!-- Navbar: ~10vh, three sections -->
    <nav class="navbar" role="navigation" aria-label="main navigation">
        <div class="navbar-brand">
            <a class="navbar-item" href="/"><strong>DA3 Portal</strong></a>
        </div>
        <div class="navbar-menu">
            <div class="navbar-start">
                <span class="navbar-item">
                    Uptime: {{ uptime_hhmm }} | Remaining: {{ remaining_hhmm }}
                </span>
            </div>
            <div class="navbar-end">
                <a class="navbar-item" href="/">Home</a>
                <a class="navbar-item" href="/gallery">Gallery</a>
            </div>
        </div>
    </nav>

    <!-- Main section: flex column, fills remaining height -->
    <div class="main-section">

        <!-- Top row: forms (40% height) -->
        <div class="top-row">

            <!-- Left area (flex: 1): two equal sub-columns, one per form -->
            <div class="top-row-left">

                <!-- Sub-column 1: Download by URL -->
                <div class="sub-column">
                    <div class="box" style="height: 100%; overflow-y: auto;">
                        <h2 class="title is-5">Download by URL</h2>
                        <form method="post" action="{{ url_for('upload_url') }}">
                            <div class="field">
                                <label class="label" for="method">Method</label>
                                <div class="control">
                                    <div class="select">
                                        <select name="method" id="method" onchange="updateUrlMethodUI()">
                                            <option value="wget" {% if method == 'wget' %}selected{% endif %}>wget</option>
                                            <option value="google-drive" {% if method == 'google-drive' %}selected{% endif %}>google-drive</option>
                                            <option value="mega.nz" {% if method == 'mega.nz' %}selected{% endif %}>mega.nz</option>
                                        </select>
                                    </div>
                                </div>
                            </div>
                            <div class="field">
                                <label class="label" for="url">URL</label>
                                <div class="control">
                                    <input class="input" type="url" name="url" id="url"
                                           placeholder="https://example.com/file.jpg"
                                           value="{{ url_value }}" required>
                                </div>
                            </div>
                            <div class="field" id="filename_row" style="display: none;">
                                <label class="label" for="filename">Filename <span class="small">(required for google-drive / mega.nz)</span></label>
                                <div class="control">
                                    <input class="input" type="text" name="filename" id="filename"
                                           placeholder="output filename"
                                           value="{{ filename_value }}"
                                           disabled>
                                </div>
                            </div>
                            <div class="field">
                                <div class="control">
                                    <button class="button is-primary" type="submit">Download</button>
                                </div>
                            </div>
                        </form>
                    </div>
                </div>

                <!-- Sub-column 2: File Upload -->
                <div class="sub-column">
                    <div class="box" style="height: 100%; overflow-y: auto;">
                        <h2 class="title is-5">Upload a File</h2>
                        <form method="post" action="{{ url_for('upload_file') }}" enctype="multipart/form-data">
                            <div class="field">
                                <div class="file has-name is-fullwidth">
                                    <label class="file-label">
                                        <input class="file-input" type="file" name="file" id="file-input" required>
                                        <span class="file-cta">
                                            <span class="file-label">Choose file…</span>
                                        </span>
                                        <span class="file-name" id="file-name-display">
                                            No file chosen
                                        </span>
                                    </label>
                                </div>
                            </div>
                            <div class="field">
                                <div class="control">
                                    <button class="button is-primary" type="submit">Upload</button>
                                </div>
                            </div>
                        </form>
                    </div>
                </div>

            </div><!-- /.top-row-left -->

            <!-- Right column 40%: MEGA credentials + Settings -->
            <div class="column col-40">
                <div class="box" style="height: 100%; overflow-y: auto;">
                    <h2 class="title is-5">MEGA Credentials &amp; Settings</h2>

                    <form method="post" action="{{ url_for('set_mega_credentials_route') }}">
                        <div class="field">
                            <label class="label" for="mega_email">Email</label>
                            <div class="control">
                                <input class="input" type="email" id="mega_email" name="mega_email"
                                       value="{{ mega_email }}" placeholder="your@email.com" required>
                            </div>
                        </div>
                        <div class="field">
                            <label class="label" for="mega_password">Password</label>
                            <div class="control">
                                <input class="input" type="password" id="mega_password" name="mega_password"
                                       placeholder="••••••••" required>
                            </div>
                        </div>
                        <div class="field">
                            <div class="control">
                                <button class="button is-info" type="submit">Connect to MEGA</button>
                            </div>
                        </div>
                    </form>

                    <p class="small mt-2">
                        Status:
                        {% if mega_cred_set %}
                            <span class="has-text-success">Connected</span>
                        {% else %}
                            <span class="has-text-danger">Not connected</span>
                        {% endif %}
                    </p>

                    <hr class="my-4">

                    <form method="post" action="{{ url_for('set_target_fps_route') }}">
                        <div class="field">
                            <label class="label" for="target_fps">Target FPS</label>
                            <div class="control">
                                <input class="input" type="number" id="target_fps" name="target_fps"
                                       min="1" max="240" step="1" value="{{ target_fps }}" required>
                            </div>
                        </div>
                        <div class="field">
                            <div class="control">
                                <button class="button is-info" type="submit">Set</button>
                            </div>
                        </div>
                    </form>

                    <form method="post" action="{{ url_for('set_device_count_route') }}">
                        <div class="field">
                            <label class="label" for="device_count">Device Count</label>
                            <div class="control">
                                <input class="input" type="number" id="device_count" name="device_count"
                                       min="1" max="8" step="1" value="{{ device_count }}" required>
                            </div>
                        </div>
                        <div class="field">
                            <div class="control">
                                <button class="button is-info" type="submit">Set</button>
                            </div>
                        </div>
                    </form>
                </div>
            </div>

        </div><!-- /.top-row -->

        <!-- Bottom row: queue status (30%) + terminal (70%) -->
        <div class="bottom-row">

            <!-- Left column: Queue status (30%) -->
            <div class="column col-30">
                <div class="box">
                    <h2 class="title is-5">Queue Status</h2>
                    <div class="scrollable-content">
                        <div class="content">
                            <p>
                                <strong>Video queue:</strong>
                                {{ video_queue_len }}{% if video_queue_max %} / {{ video_queue_max }}{% endif %}
                            </p>
                            {% if video_queue %}
                                <ol>
                                {% for item in video_queue %}
                                    <li>
                                        <a href="{{ url_for('get_upload', filename=item['stored_name']) }}">{{ item['stored_name'] }}</a>
                                        ({{ item['size_mb'] }} MB)
                                    </li>
                                {% endfor %}
                                </ol>
                            {% else %}
                                <p class="small">No videos queued yet.</p>
                            {% endif %}

                            <p>
                                <strong>FPS adjusted queue:</strong>
                                {{ fps_queue_len }}{% if fps_queue_max %} / {{ fps_queue_max }}{% endif %}
                                (pending jobs: {{ fps_jobs_pending }})
                            </p>
                            {% if fps_queue %}
                                <ol>
                                {% for item in fps_queue %}
                                    <li>
                                        <a href="{{ url_for('get_upload', filename=item['stored_name']) }}">{{ item['stored_name'] }}</a>
                                        ({{ item['size_mb'] }} MB, {{ item['target_fps'] }} fps)
                                    </li>
                                {% endfor %}
                                </ol>
                            {% else %}
                                <p class="small">No FPS adjusted videos yet.</p>
                            {% endif %}

                            <p>
                                <strong>DA3 ready:</strong> {{ da3_ready }} |
                                <strong>Tasks:</strong> {{ da3_task_queue_len }}{% if da3_task_queue_max %} / {{ da3_task_queue_max }}{% endif %}
                                (pending videos: {{ da3_task_pending_videos }})
                            </p>
                            <p class="small">Peek: <a href="/api/da3_task_queue">/api/da3_task_queue</a></p>
                        </div>
                    </div>
                </div>
            </div>

            <!-- Right column: Terminal (70%) -->
            <div class="column col-70">
                <div class="box">
                    <h2 class="title is-5">Web Terminal</h2>
                    <p class="small">Call <code>web_print('hello')</code> from notebook cells to print here.</p>
                    <div class="scrollable-content">
                        <pre id="terminal" class="terminal-pre">{{ terminal_text|e }}</pre>
                    </div>
                </div>
            </div>

        </div><!-- /.bottom-row -->

    </div><!-- /.main-section -->

    <script>
        /* ── URL method UI ───────────────────────────────── */
        function updateUrlMethodUI() {
            var method = document.getElementById('method');
            var row    = document.getElementById('filename_row');
            var input  = document.getElementById('filename');
            if (!method || !row || !input) return;
            var needs = (method.value === 'google-drive' || method.value === 'mega.nz');
            row.style.display  = needs ? 'block' : 'none';
            input.required     = needs;
            input.disabled     = !needs;
        }

        /* ── File-input label ────────────────────────────── */
        document.addEventListener('DOMContentLoaded', function () {
            var fileInput   = document.getElementById('file-input');
            var fileDisplay = document.getElementById('file-name-display');
            if (fileInput && fileDisplay) {
                fileInput.addEventListener('change', function () {
                    fileDisplay.textContent = this.files.length
                        ? this.files[0].name
                        : 'No file chosen';
                });
            }
        });

        /* ── Toast notifications ─────────────────────────── */
        function showToast(html, colorClass) {
            var container = document.getElementById('toast-container');
            var toast = document.createElement('div');
            toast.className = 'toast-item notification ' + (colorClass || 'is-info') + ' is-light';

            // Close button
            var btn = document.createElement('button');
            btn.className = 'delete';
            btn.setAttribute('aria-label', 'Close');
            btn.addEventListener('click', function () { dismissToast(toast); });
            toast.appendChild(btn);

            // Message content
            var span = document.createElement('span');
            span.innerHTML = html;
            toast.appendChild(span);

            container.appendChild(toast);

            // Trigger enter animation on next frame
            requestAnimationFrame(function () {
                requestAnimationFrame(function () {
                    toast.classList.add('toast-visible');
                });
            });

            // Auto-dismiss after 5 s
            setTimeout(function () { dismissToast(toast); }, 5000);
        }

        function dismissToast(toast) {
            if (!toast || toast.dataset.dismissing) return;
            toast.dataset.dismissing = '1';
            toast.classList.remove('toast-visible');
            toast.classList.add('toast-hiding');
            setTimeout(function () { if (toast.parentNode) toast.parentNode.removeChild(toast); }, 300);
        }

        /* ── Terminal auto-refresh ───────────────────────── */
        async function refreshTerminal() {
            var el = document.getElementById('terminal');
            if (!el) return;
            try {
                var res  = await fetch('/api/terminal');
                var data = await res.json();
                var shouldStick = (el.scrollTop + el.clientHeight + 30) >= el.scrollHeight;
                el.textContent = data.text || '';
                if (shouldStick) { el.scrollTop = el.scrollHeight; }
            } catch (e) {
                el.textContent = 'Error loading terminal: ' + e;
            }
        }

        /* ── Boot ────────────────────────────────────────── */
        document.addEventListener('DOMContentLoaded', function () {
            updateUrlMethodUI();
            refreshTerminal();
            setInterval(refreshTerminal, 1000);

            /* Fire server-side flash messages as toasts */
            {% if message %}
                showToast({{ message | tojson }}, 'is-info');
            {% endif %}
            {% if uploaded_url %}
                showToast('Saved: <a href="{{ uploaded_url }}">{{ uploaded_url }}</a>', 'is-success');
            {% endif %}
        });
    </script>
</body>
</html>
"""


_GALLERY_HTML = """
<!doctype html>
<title>Gallery - Flask + ngrok demo</title>
<style>
body { font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; margin: 20px; }
.breadcrumb { background: #f5f5f5; padding: 10px; border-radius: 4px; margin-bottom: 20px; }
.breadcrumb a { text-decoration: none; color: #0066cc; }
.file-list { list-style: none; padding: 0; }
.file-list li { padding: 8px 12px; margin: 4px 0; background: #f9f9f9; border-radius: 4px; }
.file-list li:hover { background: #f0f0f0; }
.file-list a { text-decoration: none; color: #333; }
.file-list .folder { font-weight: bold; color: #0066cc; }
.file-list .file { color: #555; }
.small { color: #666; font-size: 12px; }
.preview-img { max-width: 300px; max-height: 300px; margin: 10px 0; }
</style>
<h1>Gallery</h1>

<div class=\"breadcrumb\">
  {% for part in breadcrumb %}
    {% if loop.last %}
      <strong>{{ part.name }}</strong>
    {% else %}
      <a href=\"{{ part.url }}\">{{ part.name }}</a> / 
    {% endif %}
  {% endfor %}
</div>

{% if folders %}
  <h2>Folders</h2>
  <ul class=\"file-list\">
    {% for folder in folders %}
      <li><a href=\"{{ folder.url }}\" class=\"folder\">ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã‚Â {{ folder.name }}</a></li>
    {% endfor %}
  </ul>
{% endif %}

{% if files %}
  <h2>Files</h2>
  <ul class=\"file-list\">
    {% for file in files %}
      <li>
        <a href=\"{{ file.url }}\" class=\"file\">ÃƒÂ°Ã…Â¸Ã¢â‚¬Å“Ã¢â‚¬Å¾ {{ file.name }}</a>
        {% if file.size_mb %}<span class=\"small\">({{ file.size_mb }} MB)</span>{% endif %}
        {% if file.is_image %}
          <br><img src=\"{{ file.url }}\" class=\"preview-img\" alt=\"{{ file.name }}\">
        {% endif %}
      </li>
    {% endfor %}
  </ul>
{% endif %}

{% if not folders and not files %}
  <p class=\"small\">This folder is empty.</p>
{% endif %}

<hr>
<p><a href=\"/\">ÃƒÂ¢Ã¢â‚¬Â Ã‚Â Back to Home</a></p>
"""


def _home_context(**kwargs):
    elapsed = time.time() - KERNEL_START_TS
    _drain_web_print_queue(max_items=2000)
    with WEB_TERMINAL_LOCK:
        terminal_text = "\n".join(WEB_TERMINAL_BUFFER)
    with VIDEO_QUEUE_LOCK:
        video_queue = list(VIDEO_UPLOAD_QUEUE)
    with DA3_QUEUE_LOCK:
        fps_queue = list(DA3_QUEUE)
    try:
        fps_jobs_pending = DA3_JOB_QUEUE.qsize()
    except Exception:
        fps_jobs_pending = 0
    with FRAME_QUEUE_LOCK:
        da3_task_queue_len = len(FRAME_QUEUE)
        da3_task_queue_max = FRAME_QUEUE.maxlen or 0
    with FRAME_VIDEO_STATE_LOCK:
        da3_task_pending_videos = len(FRAME_VIDEO_ORDER)
    ctx = {
        "uptime_hhmm": _format_hhmm(elapsed),
        "remaining_hhmm": _format_hhmm(TWELVE_HOURS_SEC - elapsed),
        "method": "wget",
        "url_value": "",
        "filename_value": "",
        "terminal_text": terminal_text,
        "target_fps": get_target_fps(),
        "device_count": get_device_count(),
        "video_queue_len": len(video_queue),
        "video_queue_max": VIDEO_UPLOAD_QUEUE.maxlen or 0,
        "video_queue": video_queue,
        "fps_queue_len": len(fps_queue),
        "fps_queue_max": DA3_QUEUE.maxlen or 0,
        "fps_queue": fps_queue,
        "fps_jobs_pending": fps_jobs_pending,
        "da3_ready": _is_da3_set(),
        "da3_task_queue_len": da3_task_queue_len,
        "da3_task_queue_max": da3_task_queue_max,
        "da3_task_pending_videos": da3_task_pending_videos,
        "mega_cred_set": bool(isMEGACredSet),
        "mega_email": _MEGA_EMAIL if isMEGACredSet else "",
    }
    ctx.update(kwargs)
    return ctx


MAX_URL_DOWNLOAD_BYTES = 25 * 1024 * 1024 * 1024  # 25GB
MAX_URL_REDIRECTS = 3
ALLOWED_URL_METHODS = {'wget', 'google-drive', 'mega.nz'}


def _hostname_looks_public(hostname: str) -> bool:
    try:
        infos = socket.getaddrinfo(hostname, None)
    except socket.gaierror:
        return False

    for info in infos:
        ip_str = info[4][0]
        try:
            ip = ipaddress.ip_address(ip_str)
        except ValueError:
            return False

        if (
            ip.is_private
            or ip.is_loopback
            or ip.is_link_local
            or ip.is_reserved
            or ip.is_multicast
            or ip.is_unspecified
        ):
            return False

    return True


def _validate_fetch_url(parsed: urllib.parse.ParseResult) -> None:
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        raise ValueError("URL must start with http:// or https://")
    if parsed.username or parsed.password:
        raise ValueError("URLs with embedded credentials are not allowed.")
    if not parsed.hostname:
        raise ValueError("Invalid URL hostname.")

    try:
        port = parsed.port
    except ValueError as exc:
        raise ValueError("Invalid port in URL.") from exc
    if port not in {None, 80, 443}:
        raise ValueError("Only ports 80 and 443 are allowed.")

    if not _hostname_looks_public(parsed.hostname):
        raise ValueError("Refusing to fetch from private/reserved hosts.")


class _SafeRedirect(urllib.request.HTTPRedirectHandler):
    def __init__(self, max_redirects: int):
        super().__init__()
        self._remaining = int(max_redirects)

    def redirect_request(self, req, fp, code, msg, headers, newurl):
        self._remaining -= 1
        if self._remaining < 0:
            raise ValueError("Too many redirects.")

        if not urllib.parse.urlparse(newurl).scheme:
            newurl = urllib.parse.urljoin(req.full_url, newurl)

        _validate_fetch_url(urllib.parse.urlparse(newurl))
        return super().redirect_request(req, fp, code, msg, headers, newurl)


def handle_url_upload(method: str, url: str, filename: str = "") -> Path:
    method = (method or "wget").strip()
    if method not in ALLOWED_URL_METHODS:
        raise ValueError(f"Unknown method: {method}")

    url = (url or "").strip()
    if not url:
        raise ValueError("Please enter a URL.")
    if len(url) > 2048:
        raise ValueError("URL is too long.")

    parsed = urllib.parse.urlparse(url)
    _validate_fetch_url(parsed)

    safe_filename = secure_filename((filename or "").strip())
    if safe_filename and len(safe_filename) > 150:
        raise ValueError("Filename is too long.")

    # Keep MEGA fragments (#key); other methods don't need fragments.
    cleaned = url if method == "mega.nz" else parsed._replace(fragment="").geturl()

    if method == "google-drive":
        host = (parsed.hostname or "").lower()
        if not (host.endswith("drive.google.com") or host.endswith("docs.google.com")):
            raise ValueError("Expected a Google Drive URL (drive.google.com / docs.google.com).")
        if not safe_filename:
            raise ValueError("Filename is required for google-drive.")

        out_name = f"gdrive_{uuid.uuid4().hex[:12]}_{safe_filename}"
        out_path = UPLOAD_DIR / out_name

        try:
            import gdown  # type: ignore
        except Exception as exc:
            raise ValueError("gdown is not installed. Run the pip install cell.") from exc

        result = None
        try:
            result = gdown.download(cleaned, str(out_path), quiet=True, fuzzy=True)
        except TypeError:
            result = gdown.download(cleaned, str(out_path), quiet=True)

        if not out_path.exists():
            if result and Path(str(result)).exists():
                Path(str(result)).replace(out_path)
            else:
                raise ValueError("Google Drive download failed.")

        if out_path.stat().st_size > MAX_URL_DOWNLOAD_BYTES:
            out_path.unlink(missing_ok=True)
            raise ValueError(f"Downloaded file too large (>{MAX_URL_DOWNLOAD_BYTES} bytes).")

        enqueue_video(out_path, source="url:google-drive")
        return out_path

    if method == "mega.nz":
        host = (parsed.hostname or "").lower()
        if not (host.endswith("mega.nz") or host.endswith("mega.co.nz")):
            raise ValueError("Expected a mega.nz URL.")
        if not safe_filename:
            raise ValueError("Filename is required for mega.nz.")

        out_name = f"mega_{uuid.uuid4().hex[:12]}_{safe_filename}"
        out_path = UPLOAD_DIR / out_name

        try:
            _patch_asyncio_coroutine()
            from mega import Mega  # type: ignore
        except Exception as exc:
            raise ValueError("mega.py is not installed. Run the pip install cell.") from exc

        client = Mega().login()
        downloaded = None
        for fn in (
            lambda: client.download_url(cleaned, dest_path=str(UPLOAD_DIR), dest_filename=out_path.name),
            lambda: client.download_url(cleaned, str(UPLOAD_DIR), out_path.name),
            lambda: client.download_url(cleaned, str(UPLOAD_DIR)),
        ):
            try:
                downloaded = fn()
                break
            except TypeError:
                continue

        if not out_path.exists() and downloaded:
            downloaded_path = Path(str(downloaded))
            if downloaded_path.exists():
                try:
                    downloaded_path.replace(out_path)
                except Exception:
                    pass

        if not out_path.exists():
            raise ValueError("MEGA download failed.")

        if out_path.stat().st_size > MAX_URL_DOWNLOAD_BYTES:
            out_path.unlink(missing_ok=True)
            raise ValueError(f"Downloaded file too large (>{MAX_URL_DOWNLOAD_BYTES} bytes).")

        enqueue_video(out_path, source="url:mega.nz")
        return out_path

    # wget
    guessed_name = safe_filename or Path(urllib.parse.unquote(parsed.path or "")).name
    guessed_name = secure_filename(guessed_name) or "download.bin"
    out_name = f"wget_{uuid.uuid4().hex[:12]}_{guessed_name}"
    out_path = UPLOAD_DIR / out_name

    opener = urllib.request.build_opener(_SafeRedirect(MAX_URL_REDIRECTS))
    req = urllib.request.Request(cleaned, headers={"User-Agent": "Flask-ngrok-demo/1.0"})

    total = 0
    try:
        with opener.open(req, timeout=10) as resp, open(out_path, "wb") as f:
            while True:
                chunk = resp.read(64 * 1024)
                if not chunk:
                    break
                total += len(chunk)
                if total > MAX_URL_DOWNLOAD_BYTES:
                    raise ValueError(f"Remote file too large (>{MAX_URL_DOWNLOAD_BYTES} bytes).")
                f.write(chunk)
    except Exception:
        try:
            out_path.unlink(missing_ok=True)
        except Exception:
            pass
        raise

    enqueue_video(out_path, source="url:wget")
    return out_path


def handle_file_upload(file_storage) -> Path:
    if file_storage is None or not getattr(file_storage, "filename", ""):
        raise ValueError("Please choose a file.")

    original_name = secure_filename(file_storage.filename) or "upload.bin"
    out_name = f"file_{uuid.uuid4().hex[:12]}_{original_name}"
    out_path = UPLOAD_DIR / out_name
    file_storage.save(out_path)
    enqueue_video(out_path, source="file-upload")
    return out_path


@app.get("/")
def home():
    return render_template_string(_HOME_HTML, **_home_context())


@app.get("/gallery")
def gallery():
    path_str = request.args.get("path", "")
    
    # Resolve the path, ensuring it's within OUTPUT_DIR
    try:
        if path_str:
            # Prevent directory traversal
            rel_path = Path(path_str).as_posix()
            if rel_path.startswith("/") or rel_path.startswith(".."):
                rel_path = rel_path.lstrip("/").lstrip(".")
            target_path = OUTPUT_DIR / rel_path
        else:
            target_path = OUTPUT_DIR
        
        target_path = target_path.resolve()
        
        # Ensure the path is within OUTPUT_DIR
        try:
            target_path.relative_to(OUTPUT_DIR.resolve())
        except ValueError:
            return jsonify(ok=False, error="Access denied"), 403
    except Exception:
        target_path = OUTPUT_DIR
    
    if not target_path.exists():
        return render_template_string(_GALLERY_HTML, 
            breadcrumb=[{"name": "output", "url": "/gallery"}],
            folders=[], files=[], message="Output directory does not exist yet. DA3 outputs will appear here after processing.")
    
    # Build breadcrumb
    breadcrumb = [{"name": "output", "url": "/gallery"}]
    try:
        rel_parts = target_path.relative_to(OUTPUT_DIR.resolve()).parts
        for i, part in enumerate(rel_parts):
            breadcrumb.append({
                "name": part,
                "url": f"/gallery?path={'/'.join(rel_parts[:i+1])}"
            })
    except ValueError:
        pass
    
    # List contents
    folders = []
    files = []
    IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".gif", ".webp", ".bmp"}
    
    try:
        for item in sorted(target_path.iterdir()):
            if item.is_dir():
                rel_path = item.relative_to(OUTPUT_DIR.resolve()).as_posix()
                folders.append({
                    "name": item.name,
                    "url": f"/gallery?path={rel_path}"
                })
            elif item.is_file():
                rel_path = item.relative_to(OUTPUT_DIR.resolve()).as_posix()
                file_info = {
                    "name": item.name,
                    "url": f"/gallery-files/{rel_path}",
                    "size_mb": round(item.stat().st_size / (1024 * 1024), 2) if item.stat().st_size > 0 else 0,
                    "is_image": item.suffix.lower() in IMAGE_EXTENSIONS
                }
                files.append(file_info)
    except PermissionError:
        pass
    
    return render_template_string(_GALLERY_HTML,
        breadcrumb=breadcrumb,
        folders=folders,
        files=files)


@app.post("/settings/target_fps")
def set_target_fps_route():
    raw = request.form.get("target_fps", "")
    try:
        fps = set_target_fps(raw)
    except Exception as exc:
        return render_template_string(_HOME_HTML, **_home_context(message=f"Invalid Target FPS: {exc}")), 400

    web_print(f"Target FPS set to {fps}")
    return render_template_string(_HOME_HTML, **_home_context(message=f"Target FPS set to {fps}"))


@app.post("/settings/device_count")
def set_device_count_route():
    raw = request.form.get("device_count", "")
    try:
        count = set_device_count(raw)
    except Exception as exc:
        return render_template_string(_HOME_HTML, **_home_context(message=f"Invalid Device Count: {exc}")), 400

    web_print(f"Device count set to {count}")
    return render_template_string(_HOME_HTML, **_home_context(message=f"Device count set to {count}"))


@app.post("/settings/mega-credentials")
def set_mega_credentials_route():
    email = request.form.get("mega_email", "").strip()
    password = request.form.get("mega_password", "").strip()
    
    if not email or not password:
        return render_template_string(_HOME_HTML, **_home_context(message="Email and password are required.")), 400
    
    if set_mega_credentials(email, password):
        return render_template_string(_HOME_HTML, **_home_context(message="MEGA.nz connected successfully."))
    else:
        return render_template_string(_HOME_HTML, **_home_context(message="Failed to connect to MEGA.nz. Check credentials.")), 400



@app.get("/api/terminal")
def terminal_api():
    _drain_web_print_queue(max_items=2000)
    with WEB_TERMINAL_LOCK:
        text = "\n".join(WEB_TERMINAL_BUFFER)
    return jsonify(ok=True, text=text, lines=len(WEB_TERMINAL_BUFFER))


@app.get("/api/video_queue")
def video_queue_api():
    with VIDEO_QUEUE_LOCK:
        items = list(VIDEO_UPLOAD_QUEUE)
        max_len = VIDEO_UPLOAD_QUEUE.maxlen or 0
    return jsonify(ok=True, count=len(items), max=max_len, items=items)


@app.post("/api/video_queue/pop")
def video_queue_pop_api():
    item = video_queue_pop()
    if item:
        web_print(f"Dequeued video: {item.get('stored_name')}")
    return jsonify(ok=True, item=item)


@app.post("/api/video_queue/clear")
def video_queue_clear_api():
    with VIDEO_QUEUE_LOCK:
        VIDEO_UPLOAD_QUEUE.clear()
    web_print("Video queue cleared.")
    return jsonify(ok=True)


@app.get("/api/fps_adjusted_queue")
@app.get("/api/da3_queue")
def fps_adjusted_queue_api():
    with DA3_QUEUE_LOCK:
        items = list(DA3_QUEUE)
        max_len = DA3_QUEUE.maxlen or 0
    try:
        pending = DA3_JOB_QUEUE.qsize()
    except Exception:
        pending = 0
    return jsonify(ok=True, count=len(items), max=max_len, pending=pending, items=items)


@app.post("/api/fps_adjusted_queue/pop")
@app.post("/api/da3_queue/pop")
def fps_adjusted_queue_pop_api():
    item = da3_queue_pop()
    if item:
        web_print(f"Dequeued FPS adjusted video: {item.get('stored_name')}")
    return jsonify(ok=True, item=item)


@app.post("/api/fps_adjusted_queue/clear")
@app.post("/api/da3_queue/clear")
def fps_adjusted_queue_clear_api():
    with DA3_QUEUE_LOCK:
        DA3_QUEUE.clear()
    web_print("FPS adjusted queue cleared.")
    return jsonify(ok=True)


@app.get("/api/da3_task_queue")
def da3_task_queue_api():
    limit = request.args.get("limit", "200")
    items = da3_task_queue_peek(limit)
    with FRAME_QUEUE_LOCK:
        count = len(FRAME_QUEUE)
        max_len = FRAME_QUEUE.maxlen or 0
    with FRAME_VIDEO_STATE_LOCK:
        pending_videos = len(FRAME_VIDEO_ORDER)
    return jsonify(ok=True, da3_ready=_is_da3_set(), count=count, max=max_len, pending_videos=pending_videos, items=items)


@app.post("/api/da3_task_queue/pop")
def da3_task_queue_pop_api():
    item = da3_task_queue_pop()
    if item:
        web_print(f"Dequeued DA3 task: {item.get('video_file_url')} frame {item.get('frame_no')}")
    return jsonify(ok=True, item=item)


@app.post("/api/da3_task_queue/clear")
def da3_task_queue_clear_api():
    with FRAME_QUEUE_LOCK:
        FRAME_QUEUE.clear()
    with FRAME_VIDEO_STATE_LOCK:
        FRAME_VIDEO_STATE.clear()
        FRAME_VIDEO_ORDER.clear()
    FRAME_QUEUE_WAKE_EVENT.set()
    web_print("Frame queue cleared.")
    return jsonify(ok=True)


@app.post("/upload/url")
def upload_url():
    method = request.form.get("method", "wget")
    url = request.form.get("url", "")
    filename = request.form.get("filename", "")

    try:
        saved_path = handle_url_upload(method, url, filename)
    except Exception as exc:
        return render_template_string(
            _HOME_HTML,
            **_home_context(
                message=f"URL fetch failed: {exc}",
                method=method,
                url_value=url,
                filename_value=filename,
            ),
        ), 400

    return render_template_string(
        _HOME_HTML,
        **_home_context(
            message=f"URL downloaded via {method}.",
            uploaded_url=url_for("get_upload", filename=saved_path.name),
            method=method,
            url_value=url,
            filename_value=filename,
        ),
    )


@app.post("/upload/file")
def upload_file():
    try:
        saved_path = handle_file_upload(request.files.get("file"))
    except Exception as exc:
        return render_template_string(_HOME_HTML, **_home_context(message=f"File upload failed: {exc}")), 400

    return render_template_string(
        _HOME_HTML,
        **_home_context(
            message="File uploaded.",
            uploaded_url=url_for("get_upload", filename=saved_path.name),
        ),
    )


@app.get("/uploads/<path:filename>")
def get_upload(filename: str):
    return send_from_directory(str(UPLOAD_DIR), filename)


@app.get("/frames/<path:filename>")
def get_frame(filename: str):
    return send_from_directory(str(FRAMES_DIR), filename)


@app.get("/gallery-files/<path:filename>")
def get_gallery_file(filename: str):
    # Prevent directory traversal
    safe_path = Path(filename).as_posix()
    if safe_path.startswith("/") or safe_path.startswith(".."):
        return jsonify(ok=False, error="Invalid path"), 400
    
    file_path = OUTPUT_DIR / filename
    try:
        file_path = file_path.resolve()
        file_path.relative_to(OUTPUT_DIR.resolve())
    except (ValueError, Exception):
        return jsonify(ok=False, error="File not found"), 404
    
    if not file_path.exists():
        return jsonify(ok=False, error="File not found"), 404
    
    return send_from_directory(str(OUTPUT_DIR), filename)


@app.errorhandler(413)
def too_large(_exc):
    return render_template_string(_HOME_HTML, **_home_context(message="Upload too large (max 25GB).")), 413


@app.get("/api/ping")
def ping():
    return jsonify(ok=True, message="pong", ts=time.time())


@app.get("/api/echo")
def echo():
    return jsonify(ok=True, msg=request.args.get("msg", ""))


HOST = "127.0.0.1"
PORT = _pick_port(HOST, preferred=5000)

server = _ServerThread(app, HOST, PORT)
server.start()

web_print(f"Flask running locally: http://{HOST}:{PORT}")
print(f"Flask running locally: http://{HOST}:{PORT}")

Flask running locally: http://127.0.0.1:5000


## Ngrok

In [5]:
import os
from pyngrok import ngrok as ngrok_api

# If you re-run this cell, close old tunnels first.
try:
    ngrok_api.kill()
except Exception:
    pass

# Option A: set an env var before launching Jupyter (recommended)
#   Windows (PowerShell): $env:NGROK_AUTHTOKEN="..."
#   macOS/Linux (bash):  export NGROK_AUTHTOKEN="..."
# Option B: paste your token here (not recommended for shared notebooks)
token = os.environ.get("NGROK_AUTHTOKEN", "").strip() or ""

if token:
    ngrok_api.set_auth_token(token)

tunnel = ngrok_api.connect(PORT)
public_url = tunnel.public_url

print("ngrok public URL:", public_url)
print("Try:", public_url + "/api/ping")

ngrok public URL: https://ba2d-136-119-145-128.ngrok-free.app                                       
Try: https://ba2d-136-119-145-128.ngrok-free.app/api/ping


# Conda Setup

In [6]:
web_print("Setting up conda")

In [7]:
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!chmod +x miniconda.sh
!bash miniconda.sh -b -p /miniconda

!. /miniconda/etc/profile.d/conda.sh && conda init
!. /miniconda/etc/profile.d/conda.sh && conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main && conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!. /miniconda/etc/profile.d/conda.sh && conda create -n my_env python=3.12 -y && conda activate my_env

--2026-03-16 11:42:07--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.32.241, 104.16.191.158, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.32.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 162067098 (155M) [application/octet-stream]
Saving to: ‘miniconda.sh’

miniconda.sh        100%[===================>] 154.56M   193MB/s    in 0.8s    

2026-03-16 11:42:08 (193 MB/s) - ‘miniconda.sh’ saved [162067098/162067098]

PREFIX=/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... 

127.0.0.1 - - [16/Mar/2026 11:42:17] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:18] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:18] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [16/Mar/2026 11:42:19] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:20] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:21] "GET /api/terminal HTTP/1.1" 200 -


done


127.0.0.1 - - [16/Mar/2026 11:42:22] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:23] "GET /api/terminal HTTP/1.1" 200 -


installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /miniconda
no change     /miniconda/condabin/conda
no change     /miniconda/bin/conda
no change     /miniconda/bin/conda-env
no change     /miniconda/bin/activate
no change     /miniconda/bin/deactivate
no change     /miniconda/etc/profile.d/conda.sh
no change     /miniconda/etc/fish/conf.d/conda.fish
no change     /miniconda/shell/condabin/Conda.psm1
no change     /miniconda/shell/condabin/conda-hook.ps1
no change     /miniconda/lib/python3.13/site-packages/xontrib/conda.xsh
no change     /miniconda/etc/profile.d/conda.csh
modified      /root/.bashrc

==> For changes to take effect, close and re-open your current shell. <==



...

127.0.0.1 - - [16/Mar/2026 11:42:37] "GET /api/terminal HTTP/1.1" 200 -


/ 

127.0.0.1 - - [16/Mar/2026 11:42:38] "GET /api/terminal HTTP/1.1" 200 -


done
#
# To activate this environment, use
#
#     $ conda activate my_env
#
# To deactivate an active environment, use
#
#     $ conda deactivate



127.0.0.1 - - [16/Mar/2026 11:42:39] "GET /api/terminal HTTP/1.1" 200 -


In [8]:
%%writefile on_conda
#!/bin/bash

src=". /miniconda/etc/profile.d/conda.sh"
env="conda activate my_env"
cmd="$src && $env && $@"
eval $cmd

Writing on_conda


In [9]:
!chmod -x on_conda

import os
os.environ["konda"] = "bash on_conda"
!$konda conda --version

isCondaSet = True

127.0.0.1 - - [16/Mar/2026 11:42:40] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:41] "GET /api/terminal HTTP/1.1" 200 -


conda 26.1.1


In [ ]:
if isCondaSet:
    web_print("Conda is installed succesfully.")
else:
    web_print("Unable to set conda")

127.0.0.1 - - [16/Mar/2026 11:42:42] "GET /api/terminal HTTP/1.1" 200 -


# DA3 Setup

In [11]:
web_print("Setting up DA3")

In [12]:
!git clone https://github.com/ByteDance-Seed/Depth-Anything-3.git

Cloning into 'Depth-Anything-3'...


127.0.0.1 - - [16/Mar/2026 11:42:43] "GET /api/terminal HTTP/1.1" 200 -


remote: Enumerating objects: 356, done.
remote: Counting objects: 100% (241/241), done.
remote: Compressing objects: 100% (164/164), done.
remote: Total 356 (delta 139), reused 77 (delta 77), pack-reused 115 (from 1)
Receiving objects: 100% (356/356), 22.66 MiB | 38.81 MiB/s, done.
Resolving deltas: 100% (156/156), done.


127.0.0.1 - - [16/Mar/2026 11:42:44] "GET /api/terminal HTTP/1.1" 200 -


In [13]:
!$konda pip install xformers torch\>=2 torchvision -q
!$konda pip install -e Depth-Anything-3/. -q # Basic 
!$konda pip install --no-build-isolation git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70 -q# for gaussian head
!$konda pip install -e "Depth-Anything-3/.[app]" -q # Gradio, python>=3.10
!$konda pip install -e "Depth-Anything-3/.[all]" -q # ALL

127.0.0.1 - - [16/Mar/2026 11:42:45] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:46] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:47] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:48] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:49] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:50] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:51] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:52] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:53] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:54] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:55] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:56] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:57] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:42:58] "GET /api/terminal HTTP/1.1

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done


127.0.0.1 - - [16/Mar/2026 11:46:13] "GET /api/terminal HTTP/1.1" 200 -


  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done


127.0.0.1 - - [16/Mar/2026 11:46:14] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:46:15] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:46:16] "GET /api/terminal HTTP/1.1" 200 -


...

127.0.0.1 - - [16/Mar/2026 11:47:57] "GET /api/terminal HTTP/1.1" 200 -


  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done


127.0.0.1 - - [16/Mar/2026 11:47:58] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:47:59] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:00] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:01] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:02] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:03] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:04] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:05] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:06] "GET /api/terminal HTTP/1.1" 200 -


  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'gsplat' when getting requirements to build wheel


127.0.0.1 - - [16/Mar/2026 11:48:07] "GET /api/terminal HTTP/1.1" 200 -


In [ ]:
# Install required packages
!pip install pyngrok fastapi uvicorn -q
isDA3Set = True

127.0.0.1 - - [16/Mar/2026 11:48:08] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:09] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:10] "GET /api/terminal HTTP/1.1" 200 -


127.0.0.1 - - [16/Mar/2026 11:48:11] "GET /api/terminal HTTP/1.1" 200 -


In [ ]:
if isDA3Set:
    web_print("DA3 is installed succesfully.")
else:
    web_print("Unable to set DA3")

127.0.0.1 - - [16/Mar/2026 11:48:12] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:13] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:14] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:15] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:16] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:17] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:18] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:19] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:21] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:22] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:23] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:24] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:25] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:48:26] "GET /api/terminal HTTP/1.1

# Stop / cleanup

Run the next cell when you're done to stop ngrok and the Flask server.

In [ ]:
# Stop DA3 process workers
try:
    stop_da3_process_workers()
except Exception:
    pass

# Stop MEGA upload worker
try:
    stop_mega_upload_worker()
except Exception:
    pass

# Stop ngrok (if running)
try:
    from pyngrok import ngrok as ngrok_api

    if "public_url" in globals():
        try:
            ngrok_api.disconnect(public_url)
        except Exception:
            pass
    ngrok_api.kill()
except Exception:
    pass

# Stop Flask (if running)
if "server" in globals():
    try:
        server.shutdown()
    except Exception:
        pass

print("Stopped ngrok + Flask + DA3 workers + MEGA upload.")

# Check upload directory

In [ ]:
!nvidia-smi

Mon Mar 16 11:53:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

127.0.0.1 - - [16/Mar/2026 11:53:14] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:53:15] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:53:16] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:53:17] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:53:18] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:53:19] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:53:20] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:53:21] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:53:22] "GET /api/terminal HTTP/1.1" 200 -
127.0.0.1 - - [16/Mar/2026 11:53:27] "GET /api/terminal HTTP/1.1" 200 -


In [ ]:
!ls /kaggle/working/output/fpsadj_a892555d731b_gdrive_b102d144c999_08.15.01-08.20.00R000_fps1/frame_000000/